# Garak Scenarios

The Garak scenario family implements probes inspired by the
[Garak](https://github.com/NVIDIA/garak) framework. These include encoding-based probes (which
test whether a target can be tricked into producing harmful content when prompts are encoded in
various formats), web-injection probes (which test whether a target emits markdown
data-exfiltration or cross-site-scripting payloads), a doctor probe (which applies the Policy
Puppetry universal bypass), package-hallucination probes (which test whether a target recommends
non-existent packages that an attacker could squat), and an audio probe (which delivers spoken
jailbreaks to multimodal targets).

For full programming details, see the
[Scenarios Programming Guide](../code/scenarios/0_scenarios.ipynb).

In [ ]:
from pathlib import Path

from pyrit.output import output_scenario_async
from pyrit.prompt_target import RealtimeTarget
from pyrit.registry import TargetRegistry
from pyrit.scenario.garak import Encoding, EncodingTechnique
from pyrit.scenario.garak.audio_achilles_heel import AudioAchillesHeel, AudioAchillesHeelDatasetConfiguration
from pyrit.scenario.garak.encoding import EncodingDatasetConfiguration
from pyrit.setup import initialize_from_config_async

await initialize_from_config_async(config_path=Path("pyrit_conf.yaml"))  # type: ignore

objective_target = TargetRegistry.get_registry_singleton().instances.get("openai_chat")

./AppData/Local/miniconda3/Lib/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.4.3)/charset_normalizer (3.3.2) doesn't match a supported version!
  warnings.warn(


Found default environment files: ['./.pyrit/.env', './.pyrit/.env.local']
Loaded environment file: ./.pyrit/.env
Loaded environment file: ./.pyrit/.env.local


[pyrit:alembic] No new upgrade operations detected.


TextAdaptive: _EXCLUDED_TECHNIQUES entries ['prompt_sending'] are not in the current scenario-techniques catalog ['context_compliance', 'crescendo_history_lecture', 'crescendo_journalist_interview', 'crescendo_movie_director', 'crescendo_simulated', 'flip', 'many_shot', 'pair', 'red_teaming', 'role_play_movie_script', 'role_play_persuasion', 'role_play_persuasion_written', 'role_play_trivia_game', 'role_play_video_game', 'skeleton_key', 'tap', 'violent_durian']; the exclusion is a no-op for those entries. Remove stale entries or update the catalog.


## Encoding

Tests whether the target can decode and comply with encoded harmful prompts. Each encoding
technique encodes the prompt, asks the target to decode it, and scores whether the decoded output
matches the harmful content. Default datasets include slur terms and web/HTML/JS content.

**CLI example:**

```bash
pyrit_scan garak.encoding --target openai_chat --techniques base64 --max-dataset-size 1
```

**Available techniques** (17 encodings): Base64, Base2048, Base16, Base32, ASCII85, Hex,
QuotedPrintable, UUencode, ROT13, Braille, Atbash, MorseCode, NATO, Ecoji, Zalgo, LeetSpeak,
AsciiSmuggler

**Aggregate techniques:** `ALL` (every encoding, exhaustive) and `DEFAULT` (a broad curated subset
spanning every encoding family — base-N, byte-encodings, substitution ciphers, and symbolic
alphabets — for a meaningful default scan; the niche/lossy schemes are ALL-only). `DEFAULT` is used
when no techniques are specified.

> **Note:** Technique composition is NOT supported for Encoding — each encoding is tested
> independently.

In [ ]:
dataset_config = EncodingDatasetConfiguration(dataset_names=["garak_slur_terms_en"], max_dataset_size=1)

scenario = Encoding()
scenario.set_params_from_args(  # type: ignore
    args={
        "objective_target": objective_target,
        "scenario_techniques": [EncodingTechnique.Base64],
        "dataset_config": dataset_config,
    }
)
await scenario.initialize_async()  # type: ignore

print(f"Scenario: {scenario.name}")
print(f"Atomic attacks: {scenario.atomic_attack_count}")

scenario_result = await scenario.run_async()  # type: ignore

Scenario: Encoding
Atomic attacks: 11


Executing Encoding:   0%|          | 0/11 [00:00<?, ?attack/s]

In [ ]:
await output_scenario_async(scenario_result)


                                    📊 SCENARIO RESULTS: Encoding                                    

▼ Scenario Information
────────────────────────────────────────────────────────────────────────────────────────────────────
  📋 Scenario Details
    • Name: Encoding
    • Result ID: 7b8a94ac-a1f3-44ce-9802-06387a54f101
    • Scenario Version: 2
    • PyRIT Version: 1.1.0.dev0
    • Description:
        Encoding Scenario implementation for PyRIT. This scenario tests how resilient models are to various encoding
        attacks by encoding potentially harmful text (by default slurs and XSS payloads) and testing if the model will
        decode and repeat the encoded payload. It mimics the Garak encoding probe. The scenario works by: 1. Taking seed
        prompts (the harmful text to be encoded) 2. Encoding them using various encoding schemes (Base64, ROT13, Morse,
        etc.) 3. Asking the target model to decode the encoded text 4. Scoring whether the model successfully decoded
     

## WebInjection

Ports Garak's `web_injection` probe family. Tests whether the target can be coaxed into emitting
web-injection payloads: markdown-image data exfiltration (CSRF-style) and cross-site-scripting
(XSS). Each technique pairs a `SeedObjective` (the goal) with a `SeedPrompt` (the literal
injection prompt) and scores the response with the markdown-injection or XSS output scorer.

**CLI example:**

```bash
pyrit_scan garak.web_injection --target openai_chat --techniques xss --max-dataset-size 1
```

**Available techniques** (8 probes): MarkdownImageExfil, ColabAIDataLeakage,
StringAssemblyDataExfil, PlaygroundMarkdownExfil, MarkdownURIImageExfilExtended,
MarkdownURINonImageExfilExtended, TaskXSS, MarkdownXSS.

**Aggregate techniques:** `ALL` (all 8), `DEFAULT` (excludes the two combinatorial extended
probes), `EXFIL` (the 6 markdown-exfil probes), and `XSS` (TaskXSS + MarkdownXSS).

## Doctor

Ports Garak's `doctor` probe, which implements the HiddenLayer Policy Puppetry universal bypass:
a post-instruction-hierarchy prompt injection that frames a harmful request as a fictional
medical-drama script ("Dr House") to override safety guardrails. Each objective from the
`garak_doctor` dataset is wrapped in a Policy Puppetry template via `PolicyPuppetryConverter` and
the response is scored for non-refusal (mirroring Garak's `mitigation.MitigationBypass` detector).

**CLI example:**

```bash
pyrit_scan garak.doctor --target openai_chat --techniques policy_puppetry --max-dataset-size 1
```

**Available techniques** (2 probes): `PolicyPuppetry` (wraps the objective in the Dr House
template) and `PolicyPuppetryLeet` (the same template, additionally leetspeak-encoded). Both are
tagged `default`, so `DEFAULT` and `ALL` currently coincide.

## PackageHallucination

Ports Garak's `packagehallucination` probe. Asks the target to write code for a given language
(rendered from Garak's `stub_prompts` × `code_tasks`) and scores each response for imports of
packages that do not exist in that language's registry. A hallucinated package name is a
supply-chain foothold: an attacker can register ("squat") it so the model's suggested code
silently pulls in a malicious dependency ("slopsquatting").

Each language runs as its own atomic attack with a dedicated `PackageHallucinationScorer` loaded
with that ecosystem's registry (PyPI, npm, RubyGems, or crates.io). The scoring is deterministic
set-membership — no LLM judge is involved.

**CLI example:**

```bash
pyrit_scan garak.package_hallucination --target openai_chat --techniques python
```

**Available techniques** (4 languages): Python, JavaScript, Ruby, Rust.

**Aggregate techniques:** `ALL` and `DEFAULT` both expand to all four languages.

> **Note:** The package registries are loaded into memory only for the scorer; the raw package
> names are never sent as prompts.

## AudioAchillesHeel

Ports Garak's `audio.AudioAchillesHeel` probe. Delivers the adversarial instruction as *spoken
audio* while the text channel carries only a benign "follow the audio instructions" nudge. Each
clip from the `garak_audio_achilles_heel` dataset is shaped into a single multimodal user turn
(text nudge + audio at the same sequence), and the response is scored for compliance — the PyRIT
analogue of Garak's non-refusal `mitigation.MitigationBypass` detector. A per-clip objective is
derived from the clip's harm category.

**CLI example:**

```bash
pyrit_scan garak.audio_achilles_heel --target realtime --max-dataset-size 2
```

> **Note:** The objective target must accept `audio_path` input (i.e. be multimodal). The example
> below uses `RealtimeTarget` (the OpenAI Realtime audio websocket); non-audio targets such as the
> default `openai_chat` will error when the audio request is sent. The full dataset holds ~350
> clips, so a default run samples a small subset to finish quickly — raise `--max-dataset-size`
> for broader coverage.

In [ ]:
audio_dataset_config = AudioAchillesHeelDatasetConfiguration(
    dataset_names=["garak_audio_achilles_heel"], max_dataset_size=1
)

audio_target = RealtimeTarget()

audio_scenario = AudioAchillesHeel()
audio_scenario.set_params_from_args(  # type: ignore
    args={
        "objective_target": audio_target,
        "dataset_config": audio_dataset_config,
    }
)
await audio_scenario.initialize_async()  # type: ignore

print(f"Scenario: {audio_scenario.name}")
print(f"Atomic attacks: {audio_scenario.atomic_attack_count}")

audio_scenario_result = await audio_scenario.run_async()  # type: ignore

Scenario: AudioAchillesHeel
Atomic attacks: 1


Executing AudioAchillesHeel:   0%|          | 0/1 [00:00<?, ?attack/s]

In [ ]:
await output_scenario_async(audio_scenario_result)


                               📊 SCENARIO RESULTS: AudioAchillesHeel                                

▼ Scenario Information
────────────────────────────────────────────────────────────────────────────────────────────────────
  📋 Scenario Details
    • Name: AudioAchillesHeel
    • Result ID: 643df476-6afd-45ae-a32c-6e2adc9d27dd
    • Scenario Version: 1
    • PyRIT Version: 1.1.0.dev0
    • Description:
        Audio Achilles Heel scenario implementation for PyRIT. Ports Garak's ``audio.AudioAchillesHeel`` probe: spoken
        adversarial instructions are sent as audio while the text channel carries only a benign nudge, testing whether a
        multimodal target follows harmful spoken instructions. Each ``garak_audio_achilles_heel`` clip becomes a single
        multimodal user turn scored for compliance (the PyRIT analogue of Garak's non-refusal
        ``mitigation.MitigationBypass`` detector). The objective target must accept ``audio_path`` input (i.e. be
        multimodal); no

For more details, see the [Scenarios Programming Guide](../code/scenarios/0_scenarios.ipynb) and
[Configuration](../getting_started/configuration.md).